In [1]:
import pandas as pd

In [ ]:
# Importar datos
products = pd.read_csv(r"C:\Users\bella\OneDrive\Escritorio\DATA SCIENCE\PF_Data_science\data\raw\olist_products_dataset.csv")
order_items = pd.read_csv(r"C:\Users\bella\OneDrive\Escritorio\DATA SCIENCE\PF_Data_science\data\raw\olist_order_items_dataset.csv")
orders = pd.read_csv(r"C:\Users\bella\OneDrive\Escritorio\DATA SCIENCE\PF_Data_science\data\raw\olist_orders_dataset.csv")
reviews = pd.read_csv(r"C:\Users\bella\OneDrive\Escritorio\DATA SCIENCE\PF_Data_science\data\raw\olist_order_reviews_dataset.csv")

In [ ]:
# Revición preliminar de Datos
print(products.columns)
print(order_items.columns)
print(orders.columns)
print(reviews.columns)

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='str')


In [ ]:
# Número de productos Pet_shop
pet_products = products[products["product_category_name"] == "pet_shop"]
pet_products_count = pet_products["product_id"].nunique()

print("Productos únicos en pet_shop:", pet_products_count)

Productos únicos en pet_shop: 719


In [ ]:
#Número compras hay de productos de mascotas
pet_order_items = order_items.merge(
    pet_products[["product_id"]],
    on="product_id",
    how="inner"
)

pet_orders_count = pet_order_items["order_id"].nunique()

print("Órdenes con productos de mascotas:", pet_orders_count)

Órdenes con productos de mascotas: 1710


In [ ]:
# Número de reseñas
pet_reviews = reviews.merge(
    pet_order_items[["order_id"]].drop_duplicates(),
    on="order_id",
    how="inner"
)

pet_reviews_count = pet_reviews["review_id"].nunique()

print("Reseñas asociadas a compras de mascotas:", pet_reviews_count)

Reseñas asociadas a compras de mascotas: 1700


In [ ]:
# N° Clientes
pet_orders = orders.merge(
    pet_order_items[["order_id"]].drop_duplicates(),
    on="order_id",
    how="inner"
)

pet_customers_count = pet_orders["customer_id"].nunique()

print("Clientes únicos que compraron mascotas:", pet_customers_count)


Clientes únicos que compraron mascotas: 1710


In [ ]:
# ticket promedio
pet_ticket_avg = pet_order_items["price"].mean()

print("Precio promedio por ítem pet_shop:", pet_ticket_avg)

Precio promedio por ítem pet_shop: 110.0746841294299


In [ ]:
# Total de reseñas en categoria Pet Shop
total_pet_reviews = pet_reviews.shape[0]
total_pet_reviews_con_texto = (
    pet_reviews["review_comment_message"].notna() &
    (pet_reviews["review_comment_message"].str.strip() != "")
).sum()

print("Total de reseñas de pet_shop:", total_pet_reviews)
print("Total de reseñas de pet_shop con texto:", total_pet_reviews_con_texto)

Total de reseñas de pet_shop: 1703
Total de reseñas de pet_shop con texto: 650


In [ ]:
# Esplorar categorias y productos
categorias_productos = (
    products.dropna(subset=["product_category_name"])
    .groupby("product_category_name")["product_id"]
    .nunique()
    .reset_index(name="numero_productos")
    .sort_values(by="numero_productos", ascending=False)
)

categorias_productos

,product_category_name,numero_productos
13,cama_mesa_banho,3029
32,esporte_lazer,2867
54,moveis_decoracao,2657
11,beleza_saude,2444
72,utilidades_domesticas,2335
...,...,...
15,casa_conforto_2,5
37,fashion_roupa_infanto_juvenil,5
60,pc_gamer,3
67,seguros_e_servicos,2


In [ ]:
# Top de catégorias
top_10_categorias = (
    products.dropna(subset=["product_category_name"])
    .groupby("product_category_name")["product_id"]
    .nunique()
    .reset_index(name="numero_productos")
    .sort_values(by="numero_productos", ascending=False)
    .head(10)
)

top_10_categorias

,product_category_name,numero_productos
13,cama_mesa_banho,3029
32,esporte_lazer,2867
54,moveis_decoracao,2657
11,beleza_saude,2444
72,utilidades_domesticas,2335
8,automotivo,1900
44,informatica_acessorios,1639
12,brinquedos,1411
66,relogios_presentes,1329
70,telefonia,1134


In [21]:
# Unir productos con items de orden
items_con_categoria = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

# Quitar categorías nulas
items_con_categoria = items_con_categoria.dropna(subset=["product_category_name"])

# Valor por orden y categoría
valor_orden_categoria = (
    items_con_categoria
    .groupby(["product_category_name", "order_id"])["price"]
    .sum()
    .reset_index(name="valor_orden_categoria")
)

# Ticket promedio por categoría
ticket_promedio_categoria = (
    valor_orden_categoria
    .groupby("product_category_name")["valor_orden_categoria"]
    .mean()
    .reset_index(name="ticket_promedio")
    .sort_values(by="ticket_promedio", ascending=False)
)

In [8]:
print(ticket_promedio_categoria.to_string(index=False))

                         product_category_name  ticket_promedio
                                           pcs      1231.840497
                   portateis_casa_forno_e_cafe       632.609467
                            eletrodomesticos_2       484.263846
                     agro_industria_e_comercio       398.519066
                         instrumentos_musicais       304.934522
                               eletroportateis       302.616794
 portateis_cozinha_e_preparadores_de_alimentos       283.466429
                                telefonia_fixa       274.576037
              construcao_ferramentas_seguranca       242.781557
                                  climatizacao       217.489960
                             moveis_escritorio       215.208720
                            relogios_presentes       214.261323
                                 moveis_quarto       210.829263
             construcao_ferramentas_construcao       193.419238
                                      pc

In [ ]:
# Relación Catégoria Porductos y Ticket
resumen_categorias = categorias_productos.merge(
    ticket_promedio_categoria,
    on="product_category_name",
    how="left"
).sort_values(by="numero_productos", ascending=False)

resumen_categorias

,product_category_name,numero_productos,ticket_promedio
0,cama_mesa_banho,3029,110.118794
1,esporte_lazer,2867,127.985618
2,moveis_decoracao,2657,113.159015
3,beleza_saude,2444,142.449224
4,utilidades_domesticas,2335,107.452186
...,...,...,...
68,casa_conforto_2,5,31.677917
69,fashion_roupa_infanto_juvenil,5,71.231250
70,pc_gamer,3,193.243750
71,seguros_e_servicos,2,141.645000


In [ ]:
# top 60 de Productos y ticket
resumen_categorias_top20 = (
    categorias_productos.merge(
        ticket_promedio_categoria,
        on="product_category_name",
        how="left"
    )
    .rename(columns={
        "product_category_name": "categoria",
        "numero_productos": "numero_productos",
        "ticket_promedio": "ticket_promedio"
    })
    .sort_values(by="numero_productos", ascending=False)
    .head(60)
    .reset_index(drop=True)
)

resumen_categorias_top20.index = resumen_categorias_top20.index + 1
resumen_categorias_top20["ticket_promedio"] = resumen_categorias_top20["ticket_promedio"].round(2)

resumen_categorias_top20

,categoria,numero_productos,ticket_promedio
1,cama_mesa_banho,3029,110.12
2,esporte_lazer,2867,127.99
3,moveis_decoracao,2657,113.16
4,beleza_saude,2444,142.45
5,utilidades_domesticas,2335,107.45
6,automotivo,1900,152.10
7,informatica_acessorios,1639,136.34
8,brinquedos,1411,124.54
9,relogios_presentes,1329,214.26
10,telefonia,1134,77.08


In [15]:
# Diccionario de agrupación
mapa_grupos = {
    "cama_mesa_banho": "Hogar y decoracion",
    "moveis_decoracao": "Hogar y decoracion",
    "utilidades_domesticas": "Hogar y decoracion",
    "casa_conforto": "Hogar y decoracion",
    "moveis_sala": "Hogar y decoracion",

    "beleza_saude": "Belleza, salud y accesorios personales",
    "perfumaria": "Belleza, salud y accesorios personales",
    "fashion_bolsas_e_acessorios": "Belleza, salud y accesorios personales",
    "fashion_calcados": "Belleza, salud y accesorios personales",
    "relogios_presentes": "Belleza, salud y accesorios personales",

    "informatica_acessorios": "Tecnologia y entretenimiento",
    "telefonia": "Tecnologia y entretenimiento",
    "eletronicos": "Tecnologia y entretenimiento",
    "consoles_games": "Tecnologia y entretenimiento",
    "cool_stuff": "Tecnologia y entretenimiento",

    "bebes": "Familia, mascotas y recreacion",
    "brinquedos": "Familia, mascotas y recreacion",
    "pet_shop": "Familia, mascotas y recreacion",
    "papelaria": "Familia, mascotas y recreacion",
    "esporte_lazer": "Familia, mascotas y recreacion"
}

# Crear columna de grupo
products["grupo_categoria"] = products["product_category_name"].map(mapa_grupos)

# Ver resultado
products[["product_id", "product_category_name", "grupo_categoria"]].head(20)

,product_id,product_category_name,grupo_categoria
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,"Belleza, salud y accesorios personales"
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,NaN
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,"Familia, mascotas y recreacion"
3,cef67bcfe19066a932b7673e239eb23d,bebes,"Familia, mascotas y recreacion"
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,Hogar y decoracion
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,NaN
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,Tecnologia y entretenimiento
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,Hogar y decoracion
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,NaN
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,"Familia, mascotas y recreacion"


In [ ]:
# Total Productos por Grupo
productos_por_grupo = (
    products.dropna(subset=["grupo_categoria"])
    .groupby("grupo_categoria")["product_id"]
    .nunique()
    .reset_index(name="numero_productos")
    .sort_values(by="numero_productos", ascending=False)
)

productos_por_grupo

,grupo_categoria,numero_productos
2,Hogar y decoracion,8288
1,"Familia, mascotas y recreacion",6765
0,"Belleza, salud y accesorios personales",5663
3,Tecnologia y entretenimiento,4396


In [ ]:
# Categoría de producto y grupo 
categorias_con_grupo = (
    products[["product_category_name", "grupo_categoria"]]
    .dropna(subset=["grupo_categoria"])
    .drop_duplicates()
    .sort_values(by=["grupo_categoria", "product_category_name"])
    .reset_index(drop=True)
)

categorias_con_grupo

,product_category_name,grupo_categoria
0,beleza_saude,"Belleza, salud y accesorios personales"
1,fashion_bolsas_e_acessorios,"Belleza, salud y accesorios personales"
2,fashion_calcados,"Belleza, salud y accesorios personales"
3,perfumaria,"Belleza, salud y accesorios personales"
4,relogios_presentes,"Belleza, salud y accesorios personales"
5,bebes,"Familia, mascotas y recreacion"
6,brinquedos,"Familia, mascotas y recreacion"
7,esporte_lazer,"Familia, mascotas y recreacion"
8,papelaria,"Familia, mascotas y recreacion"
9,pet_shop,"Familia, mascotas y recreacion"


In [ ]:
# Grupo y ticket
items_con_grupo = order_items.merge(
    products[["product_id", "product_category_name", "grupo_categoria"]],
    on="product_id",
    how="left"
)

items_con_grupo = items_con_grupo.dropna(subset=["grupo_categoria"])

valor_orden_grupo = (
    items_con_grupo
    .groupby(["grupo_categoria", "order_id"])["price"]
    .sum()
    .reset_index(name="valor_orden_grupo")
)

ticket_promedio_grupo = (
    valor_orden_grupo
    .groupby("grupo_categoria")["valor_orden_grupo"]
    .mean()
    .reset_index(name="ticket_promedio_grupo")
    .sort_values(by="ticket_promedio_grupo", ascending=False)
)

ticket_promedio_grupo["ticket_promedio_grupo"] = ticket_promedio_grupo["ticket_promedio_grupo"].round(2)

ticket_promedio_grupo

,grupo_categoria,ticket_promedio_grupo
0,"Belleza, salud y accesorios personales",154.20
1,"Familia, mascotas y recreacion",126.03
3,Tecnologia y entretenimiento,120.88
2,Hogar y decoracion,112.80


In [ ]:
# Resumen
resumen_grupos = productos_por_grupo.merge(
    ticket_promedio_grupo,
    on="grupo_categoria",
    how="left"
)

resumen_grupos

,grupo_categoria,numero_productos,ticket_promedio_grupo
0,Hogar y decoracion,8288,112.80
1,"Familia, mascotas y recreacion",6765,126.03
2,"Belleza, salud y accesorios personales",5663,154.20
3,Tecnologia y entretenimiento,4396,120.88
